**Authors:** Pablo Rodríguez Elvira, Adrián Segura Onorato

# Design and implementation



In [18]:
import altair as alt
import pandas as pd
from pathlib import Path
import numpy as np

input_file = Path("simpsons_episodes_clean.csv")
df = pd.read_csv(input_file)

## 1. How have the ratings evolved over time?

In [19]:
chart_1 = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X("season_episode:O", title="Episodes"),
    y=alt.Y("imdb_rating:Q", title="IMDb rating", scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip("season_episode:O", title="Season"),
        alt.Tooltip("imdb_rating:Q", title="Rating", format=".1f")
    ]
).properties(
    title="Evolution of ratings over time",
    height=350
).configure_title(
    fontSize=20
)

chart_1

alt.Chart(...)

In [ ]:
base = alt.Chart(df).encode(
    x=alt.X(
        "season:O",
        title="Season number",
        sort=sorted(df["season"].astype(int).unique()),
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "mean(imdb_rating):Q",
        title="Average IMDb rating",
        scale=alt.Scale(zero=False)
    ),
    tooltip=[
        alt.Tooltip("season:O", title="Season"),
        alt.Tooltip("mean(imdb_rating):Q", title="Average IMDb rating", format=".2f")
    ]
)

line = base.mark_line(strokeWidth=3)

points = base.mark_point(
    size=70,
    filled=True
)

labels_df = df[df["season"].astype(int).isin([1, 7, 12, 27])]

labels = alt.Chart(labels_df).mark_text(
    dy=-12,
    fontSize=11,
    fontWeight="bold"
).encode(
    x=alt.X("season:O", sort=sorted(df["season"].astype(int).unique())),
    y=alt.Y("mean(imdb_rating):Q"),
    text=alt.Text("mean(imdb_rating):Q", format=".1f")
)

chart_2 = (line + points + labels).properties(
    title="Average IMDb rating by season",
    width=650,
    height=350
).configure_title(
    fontSize=20,
    anchor="middle"
)

chart_2

alt.LayerChart(...)

In [21]:
base = alt.Chart(df).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0, orient="top")),
    y=alt.Y("number_in_season:O", title="Episode number", sort="ascending")
)

heatmap = base.mark_rect(cornerRadius=6).encode(
    color=alt.Color(
        "rating_group:N",
        title="Rating",
        scale=alt.Scale(
            domain=["<=5", "5.0-5.9", "6.0-6.9", "7.0-7.9", "8.0-8.9", ">=9.0"],
            range=["#780000", "#D73939", "#d47336", "#f9ec5f", "#3dbd3d", "#005800"]
        )
    )
)

text = base.mark_text(fontSize=11, fontWeight="bold").encode(
    text=alt.Text("imdb_rating:Q", format=".1f"),
    color=alt.condition(
        (alt.datum.imdb_rating >= 9) | (alt.datum.imdb_rating <= 5.9),
        alt.value("white"),
        alt.value("black")
    )
)

chart_3 = (heatmap + text).properties(
    width=800,
    height=800,
    title="IMDb episode ratings by season"
).configure_title(
    fontSize=20,
    anchor="middle"
)

chart_3

alt.LayerChart(...)

### Design decisions

Our first idea was to create a single line chart with all episodes ordered by season. However, since the dataset contains almost 600 episodes, the chart became too dense and it was difficult to identify any clear trend in IMDb ratings. The level of detail was too high for answering the question effectively.

For this reason, we chose two complementary views. First, a season-level line chart with average ratings summarizes the long-term evolution of the series and reduces visual clutter. This makes the general pattern easier to read: average ratings rise until around season 7 and then tend to decline. However, this view alone does not show enough episode-level detail.

To complement it, we added a heatmap of episode ratings. At first, we used a continuous red–yellow–green scale, but it lacked contrast and was not easy to interpret at a glance. We then switched to clearly separated rating bands using the derived variable `rating_group`, which makes differences between low- and high-rated episodes more visible. We also adjusted the text color inside the cells, using white on darker backgrounds and black otherwise, to improve legibility.

## 2. How have the viewers evolved over time? 

In [ ]:
base = alt.Chart(df).encode(
    x=alt.X(
        "season:O",
        title="Season",
        axis=alt.Axis(labelAngle=0, orient="top")
    ),
    y=alt.Y(
        "number_in_season:O",
        title="Episode number",
        sort="ascending"
    ),
    tooltip=[
        alt.Tooltip("season:O", title="Season"),
        alt.Tooltip("number_in_season:O", title="Episode"),
        alt.Tooltip("us_viewers_in_millions:Q", title="US viewers (millions)", format=".1f")
    ]
)

heatmap = base.mark_rect(cornerRadius=4).encode(
    color=alt.Color(
        "us_viewers_in_millions:Q",
        title="US viewers (millions)",
        scale=alt.Scale(scheme="cividis")
    )
)

text = base.mark_text(fontSize=9, fontWeight="bold").encode(
    text=alt.Text("us_viewers_in_millions:Q", format=".1f"),
    color=alt.condition(
        alt.datum.us_viewers_in_millions > 18,
        alt.value("black"),
        alt.value("white")
    )
)

chart_4 = (heatmap + text).properties(
    width=800,
    height=700,
    title="US viewers per episode across seasons"
).configure_title(
    fontSize=20,
    anchor="middle"
)

chart_4

alt.LayerChart(...)

### Design decisions

To study how viewers evolved over time, we used a heatmap showing U.S. viewers for each episode across all seasons. This keeps the same overall structure as the ratings visualization, making both analyses easy to compare, but we intentionally changed the color encoding. While the ratings chart uses discrete ranges, here we used a continuous gradient because viewers are better interpreted as a quantitative magnitude.

Columns represent seasons and rows represent episode numbers. Color intensity encodes the number of viewers in millions, and the value labels inside each cell provide exact information for each episode. This allows the user to see both the global trend and the differences within each season.

We also chose a color-blind-friendly palette, so the visualization remains accessible while still making high and low audience values easy to distinguish. This design reduces clutter and helps reveal long-term patterns across the series. The chart shows that earlier seasons had much larger audiences, while later seasons generally concentrate lower values, indicating a clear overall decline in viewers over time.

## 3. Is there a correlation between the gradings and the viewers? 

In [ ]:
corr = df["imdb_rating"].corr(df["us_viewers_in_millions"])

chart_5 = alt.Chart(df).mark_circle(size=80, opacity=0.55).encode(
    x=alt.X(
        "imdb_rating:Q",
        title="IMDb rating",
        scale=alt.Scale(domain=[4, 10])
    ),
    y=alt.Y(
        "us_viewers_in_millions:Q",
        title="US viewers (millions)",
        scale=alt.Scale(domainMin=0)
    ),
    tooltip=[
        alt.Tooltip("imdb_rating:Q", title="IMDb rating", format=".1f"),
        alt.Tooltip("us_viewers_in_millions:Q", title="Viewers", format=".2f")
    ]
)

trend = alt.Chart(df).transform_regression(
    "imdb_rating", "us_viewers_in_millions"
).mark_line(size=3, clip=True, color="red").encode(
    x=alt.X("imdb_rating:Q", scale=alt.Scale(domain=[4, 10])),
    y=alt.Y("us_viewers_in_millions:Q", scale=alt.Scale(domainMin=0))
)

corr_text = alt.Chart(
    pd.DataFrame({
        "x": [4.5],
        "y": [32.5],
        "label": [f"Corr = {corr:.3f}"]
    })
).mark_text(
    align="left",
    baseline="top",
    fontSize=14,
    fontWeight="bold"
).encode(
    x="x:Q",
    y="y:Q",
    text="label:N"
)

chart_5 = (chart_5 + trend + corr_text).properties(
    title="Correlation between ratings and viewers",
    width=600,
    height=350
).configure_title(
    fontSize=20
)

chart_5

alt.LayerChart(...)

### Design decisions:

We selected a scatter plot because it is the most appropriate chart to evaluate the relationship between two quantitative variables: `imdb_rating` and `us_viewers_in_millions`. Each point represents one episode, allowing the user to inspect the overall distribution and identify whether higher-rated episodes also tend to have more viewers.

To make the pattern easier to interpret, we added a regression line and displayed the correlation coefficient directly on the chart. These elements help users answer the question quickly, since they provide both a visual and numeric summary of the relationship. Partial transparency was applied to the points to reduce overplotting and reveal dense areas more clearly.

We considered bar charts and grouped summaries, but they would have hidden the continuous nature of the variables and made the relationship harder to assess. The scatter plot is more effective because it preserves all observations and makes weak, moderate, or strong association patterns immediately visible.

## 4. Are the number of viewers for the episodes related to the weekday they were aired?

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday","Thursday", "Friday", "Saturday", "Sunday"]

chart_6 = alt.Chart(df).mark_boxplot(size=40).encode(
    x=alt.X("weekday:N", sort=weekday_order, title="Weekday aired", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("us_viewers_in_millions:Q", title="US viewers (millions)"),
).properties(
    title="US viewers by weekday aired",
    width=600,
    height=350
).configure_title(
    fontSize=20
)

chart_6

alt.Chart(...)

In [ ]:
chart_7 = alt.Chart(df).mark_bar().encode(
    x=alt.X("weekday:N", sort=weekday_order, title="Weekday aired", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("count():Q", title="Number of episodes"),
    tooltip=[
        alt.Tooltip("weekday:N", title="Weekday"),
        alt.Tooltip("count():Q", title="Episodes")
    ]
).properties(
    title="Number of episodes by weekday",
    width=600,
    height=350
).configure_title(
    fontSize=20
)

chart_7

alt.Chart(...)

### Design decisions:
We used two views to explore this question: first, a boxplot of U.S. viewers by weekday, and then a bar chart showing how many episodes were aired on each weekday. The boxplot was useful as an initial approach because it allows comparison of the distribution of viewers across weekdays, including median, spread, and outliers.

However, after inspecting the results, we decided to add the bar chart to better understand the distribution of the data. During data cleaning, we had already noticed that most episodes were aired on Sunday, so seeing Thursday appear as the weekday with higher viewer values did not fully match our expectations. The bar chart helped explain this result by showing that the dataset is highly unbalanced: Sunday contains most of the episodes, while the other weekdays have very few observations.

This second chart is therefore essential for interpreting the boxplot correctly, since apparent differences between weekdays may be misleading when some groups are so small. We ordered weekdays in their natural calendar order to improve readability. Overall, this design helps address the question, but it also makes clear that the available data does not support a fully reliable comparison between weekdays.

## 5. Do the seasons number of viewers present any relevant pattern? 


In [ ]:
base = alt.Chart(df).encode(
    x=alt.X(
        "season:O",
        title="Season number",
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "mean(us_viewers_in_millions):Q",
        title="Average viewers (millions)",
        scale=alt.Scale(zero=False)
    ),
    tooltip=[
        alt.Tooltip("season:O", title="Season"),
        alt.Tooltip("mean(us_viewers_in_millions):Q", title="Average viewers", format=".1f")
    ]
)

line = base.mark_line(strokeWidth=3).encode()

points = base.mark_point(size=60, filled=True).encode()

labels_df = df[df["season"].isin([1, 12, 27])]

labels = alt.Chart(labels_df).mark_text(
    dy=-12,
    fontSize=11,
    fontWeight="bold"
).encode(
    x=alt.X("season:O"),
    y=alt.Y("mean(us_viewers_in_millions):Q"),
    text=alt.Text("mean(us_viewers_in_millions):Q", format=".1f")
)

chart_8 = (line + points + labels).properties(
    title="Average viewers by season",
    width=650,
    height=350
).configure_title(
    fontSize=18,
    anchor="middle"
)

chart_8

alt.LayerChart(...)

In [ ]:
chart_box = alt.Chart(df).mark_boxplot(
    size=18, 
).encode(
    x=alt.X(
        "season:O",
        title="Season number",
        sort=sorted(df["season"].unique()),
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "us_viewers_in_millions:Q",
        title="US viewers (millions)",
        scale=alt.Scale(zero=False)
    )
)

chart_line = alt.Chart(df).mark_line(
    point=alt.OverlayMarkDef(color="#1f3b5c"),
    color="#1f3b5c",
    strokeWidth=2
).encode(
    x=alt.X("season:O", sort=sorted(df["season"].unique())),
    y=alt.Y("mean(us_viewers_in_millions):Q")
)

chart_9 = (chart_box + chart_line).properties(
    title="Distribution of viewers by season",
    width=700,
    height=400
).configure_title(
    fontSize=18,
    anchor="middle"
)

chart_9

alt.LayerChart(...)

### Design decisions:

We first created `chart_8`, a line chart of average viewers by season, to capture the overall pattern in a simple and direct way. This chart is effective for identifying the main trend over time, and it clearly shows three relevant points that we decided to highlight: the maximum at the beginning of the series, the minimum in the final season, and the rebound around season 12.

After that, we looked for a second chart that could provide more information, which led us to `chart_9`. We used a boxplot by season to show the distribution of viewers across episodes, since the line chart alone only represents the average. With this chart, it becomes possible to analyze variability, spread, and outliers within each season.

At first, we were unsure whether to overlay the mean line in the boxplot. We initially tried it in red, but it drew too much attention and became the most visually dominant element. Since that line represents the same average trend already shown in `chart_8`, we changed it to a more neutral dark blue so that the distribution remained the main focus.

Together, these charts show a general decline in viewers, with some intermediate fluctuations.